In [ ]:
# Debug Memory Vector Test

This notebook reproduces the memory vector test failure and prints the exact error payload.


# Debug Memory Vector Workflow
This notebook reproduces the vector memory write/update/delete flow and captures the failing error details.

In [ ]:
import os
import sys
from pathlib import Path
import sqlite3
import traceback

ROOT = Path('/config/Nexus').resolve()
sys.path.insert(0, str(ROOT))

from plugins.memory import execute
from plugins.memory import database
from plugins.memory.actions import write as write_action
from plugins.memory.actions import update as update_action
from plugins.memory.actions import delete as delete_action
from plugins.memory.actions import search as search_action
from plugins.memory.vector_store import reset_store

In [ ]:
TMP_DB = ROOT / 'plugins' / 'memory' / 'database' / 'memory_test.db'

def set_temp_database():
    database.DATABASE_PATH = TMP_DB
    if TMP_DB.exists():
        TMP_DB.unlink()
    database.ensure_database_ready()

set_temp_database()
print('Temporary DB path:', TMP_DB)

In [ ]:
reset_store()

write_payload = {
    'title': 'Pizza Day',
    'category': 'IDEA',
    'content': 'I had delicious pizza today',
    'tags': ['food', 'happy'],
}
write_resp = write_action.write(write_payload)
print('WRITE response:', write_resp)
mem_id = write_resp.get('data', {}).get('memory_id')

In [ ]:
update_payload = {
    'memory_id': mem_id,
    'changes': {'content': 'I felt sick today'},
}
update_resp = update_action.update(update_payload)
print('UPDATE response:', update_resp)

In [ ]:
if mem_id:
    delete_resp = delete_action.delete({'memory_id': mem_id})
    print('DELETE response:', delete_resp)
else:
    print('No memory id available to delete')

In [ ]:
search_resp = search_action.search({'type': 'SQLITE', 'query': 'Pizza Day', 'limit': 5, 'include_deleted': True})
print('SQLITE SEARCH response:', search_resp)

vector_resp = search_action.search({'type': 'VECTOR', 'query': 'sick', 'limit': 5})
print('VECTOR SEARCH response:', vector_resp)

In [ ]:
try:
    assert write_resp['status'] == 'SUCCESS'
    assert update_resp['status'] == 'SUCCESS'
    assert delete_resp['status'] == 'SUCCESS'
    assert 'results' in search_resp['data']
    print('All assertions passed')
except Exception:
    traceback.print_exc()
finally:
    if TMP_DB.exists():
        TMP_DB.unlink()
    reset_store()
    print('Cleanup complete')